In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import subprocess
import sys
from pathlib import Path

package_dir = Path.cwd().resolve()
if package_dir.name == "examples":
    package_dir = package_dir.parent
else:
    repo_candidate = Path("packages/visualization/calibrated-explanations-visualization-plotly").resolve()
    if repo_candidate.exists():
        package_dir = repo_candidate

subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(package_dir)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plotly>=5.18'])


# Local Uncertainty Quadrant

This notebook demonstrates `plotly.local.uncertainty_quadrant` for a single factual local CE explanation.

The plot separates two independent dimensions:

- x-axis = absolute local impact, `abs(contribution)`
- y-axis = calibrated uncertainty width, `high - low`
- contribution sign is shown separately through the marker encoding and hover text
- high-impact / low-uncertainty points are the most reliable local drivers

The first cell installs this plugin from the local source checkout with the Plotly extra.

In [ ]:
import numpy as np
from calibrated_explanations import WrapCalibratedExplainer
from ce_visualization_plotly.plugin import register_plotly_visualization_components
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

register_plotly_visualization_components()
np.set_printoptions(precision=3, suppress=True)

## Data

Generate a binary classification dataset and split it into proper training, calibration, and query data using a 60/20/20 split. The calibration set is separate from the data used to fit the model.

In [ ]:
X, y = make_classification(
    n_samples=500,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    n_classes=2,
    random_state=0,
)

x_proper, x_holdout, y_proper, y_holdout = train_test_split(
    X,
    y,
    test_size=0.40,
    random_state=0,
    stratify=y,
)
x_cal, X_query, y_cal, y_query = train_test_split(
    x_holdout,
    y_holdout,
    test_size=0.50,
    random_state=0,
    stratify=y_holdout,
)

x_proper.shape, x_cal.shape, X_query.shape

## Fit and Calibrate

`WrapCalibratedExplainer` owns the model fitting and calibration sequence. The assertions make the fitted and calibrated states explicit before explanations are requested.

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=0)

explainer = WrapCalibratedExplainer(model)
explainer.fit(x_proper, y_proper)
assert explainer.fitted is True

explainer.calibrate(x_cal, y_cal)
assert explainer.calibrated is True

## Explain and Plot

The quadrant plot answers: which local rules are both high impact and low uncertainty? Direction is not encoded through x-position; x is always absolute impact.

In [ ]:
explanations = explainer.explain_factual(X_query)
explanations[0]

In [ ]:
explanations[0].plot(style="plotly.local.uncertainty_quadrant", show=True)